# Shot Distance Accuracy Diagnostic

**Purpose:** Investigate whether shot distances are correctly calculated, focusing on
whether the change of attacking direction between periods is properly handled.

**Key concern:** If coordinate normalization fails for period 2 (when teams switch ends),
shots in that period could have inflated or deflated distances.

**What we already know from data inspection:**
- All shot data is at schema v2 (pre-normalization-fix). Current code is v3.
- Pre-2020 seasons: NHL API returns `homeTeamDefendingSide = None`, and v2 code stored
  raw coordinates unchanged → **~50% of shots have negative x_coord and ~150 ft avg distance**
- Post-2020 seasons: API provides the field, normalization works → **~2% negative x**

In [ ]:
import os
import sys
import sqlite3

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import seaborn as sns
from scipy import stats

# Path setup per CLAUDE.md convention
for _candidate in [os.path.join(os.getcwd(), "src"),
                   os.path.join(os.getcwd(), "..", "src")]:
    _candidate = os.path.abspath(_candidate)
    if os.path.isdir(_candidate) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

from database import DATABASE_PATH

conn = sqlite3.connect(DATABASE_PATH)
conn.row_factory = sqlite3.Row
print(f"Connected to {DATABASE_PATH}")
print(f"Size: {os.path.getsize(DATABASE_PATH) / 1e6:.0f} MB")

In [ ]:
CAPS_TEAM_ID = 15

RECENT_GAME_IDS = [
    2025021044,  # WSH vs BOS, 2026-03-14 (HOME)
    2025020962,  # WSH vs UTA, 2026-03-03 (HOME)
    2025020908,  # WSH vs NSH, 2026-02-05 (HOME)
]

OLD_GAME_IDS = [
    2017021187,  # WSH vs NYR, 2018-03-28 (HOME)
    2017021244,  # WSH vs NSH, 2018-04-05 (HOME)
]

ALL_GAME_IDS = RECENT_GAME_IDS + OLD_GAME_IDS

cursor = conn.cursor()
cursor.execute("SELECT team_id, team_abbrev FROM teams")
TEAM_ABBREVS = {row["team_id"]: row["team_abbrev"] for row in cursor}
TEAM_ABBREVS[CAPS_TEAM_ID] = "WSH"  # ensure present
print(f"Loaded {len(TEAM_ABBREVS)} team abbreviations")

In [ ]:
from rink_viz import (
    draw_full_rink,
    draw_half_rink,
    plot_game_shot_chart,
    PERIOD_COLORS,
    PERIOD_LABELS,
)
print("Rink drawing imported from src/rink_viz.py")

In [ ]:
from database import load_game_shots


def game_header(shots):
    """Build a descriptive game header string."""
    if not shots:
        return "No data"
    s = shots[0]
    home = TEAM_ABBREVS.get(s["home_team_id"], str(s["home_team_id"]))
    away = TEAM_ABBREVS.get(s["away_team_id"], str(s["away_team_id"]))
    return f"{away} @ {home} — {s['game_date']} (game {s['game_id']})"


games_data = {}
for gid in ALL_GAME_IDS:
    shots = load_game_shots(conn, gid)
    games_data[gid] = shots
    print(f"Game {gid}: {len(shots)} shots — {game_header(shots)}")

## Part 1: Individual Game Shot Charts — Recent Games (Post-2020)

These games have `homeTeamDefendingSide` from the API, so normalization should be correct.
All shots are normalized so the shooting team attacks toward +x (goal at x=89).

Shots are colored by **period** — if coordinate normalization works correctly, all periods
should cluster in the offensive zone (high x values, near the goal).

In [ ]:
def plot_game_half_rink(game_id, shots, ax_chart, ax_hist):
    """Plot a half-rink shot chart and distance histogram for one game."""
    plot_game_shot_chart(ax_chart, shots, full_rink=False)
    ax_chart.set_title(game_header(shots), fontsize=10)

    periods_in_game = sorted(set(s["period"] for s in shots if s["period"] is not None))
    for period in periods_in_game:
        dists = [s["distance_to_goal"] for s in shots
                 if s["period"] == period and s["distance_to_goal"] is not None]
        if dists:
            color = PERIOD_COLORS.get(period, "gray")
            label = PERIOD_LABELS.get(period, f"P{period}")
            ax_hist.hist(dists, bins=30, alpha=0.4, color=color, label=label,
                         range=(0, 200), density=True)
    ax_hist.set_xlabel("Distance to goal (ft)")
    ax_hist.set_ylabel("Density")
    ax_hist.set_title("Distance distribution by period")
    ax_hist.legend(fontsize=8)


fig, axes = plt.subplots(len(RECENT_GAME_IDS), 2, figsize=(18, 6 * len(RECENT_GAME_IDS)))
if len(RECENT_GAME_IDS) == 1:
    axes = axes.reshape(1, -1)

for i, gid in enumerate(RECENT_GAME_IDS):
    plot_game_half_rink(gid, games_data[gid], axes[i, 0], axes[i, 1])

fig.suptitle("Recent Caps Games (post-2020) — Normalized Coordinates", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Old-Era Games (Pre-2020) — Missing `homeTeamDefendingSide`

For these games, the NHL API does not provide `homeTeamDefendingSide`. The v2 code
stored raw coordinates unchanged. Expect ~50% of shots to appear on the **wrong side
of the rink** (negative x, far from the goal at x=89).

Using a **full rink** view to show both correctly and incorrectly placed shots.

In [ ]:
def plot_game_full_rink(game_id, shots, ax_chart, ax_hist):
    """Plot a full-rink shot chart and distance histogram for one game."""
    plot_game_shot_chart(ax_chart, shots, full_rink=True)
    ax_chart.set_title(game_header(shots), fontsize=10)

    periods_in_game = sorted(set(s["period"] for s in shots if s["period"] is not None))
    for period in periods_in_game:
        dists = [s["distance_to_goal"] for s in shots
                 if s["period"] == period and s["distance_to_goal"] is not None]
        if dists:
            color = PERIOD_COLORS.get(period, "gray")
            label = PERIOD_LABELS.get(period, f"P{period}")
            ax_hist.hist(dists, bins=40, alpha=0.4, color=color, label=label,
                         range=(0, 200), density=True)
    ax_hist.set_xlabel("Distance to goal (ft)")
    ax_hist.set_ylabel("Density")
    ax_hist.set_title("Distance distribution by period (expect bimodal)")
    ax_hist.legend(fontsize=8)


fig, axes = plt.subplots(len(OLD_GAME_IDS), 2, figsize=(20, 6 * len(OLD_GAME_IDS)))
if len(OLD_GAME_IDS) == 1:
    axes = axes.reshape(1, -1)

for i, gid in enumerate(OLD_GAME_IDS):
    plot_game_full_rink(gid, games_data[gid], axes[i, 0], axes[i, 1])

fig.suptitle("Old-Era Caps Games (pre-2020) — Raw Coordinates (v2 bug)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## Part 2: Aggregate Period Comparison — Is Period 2 Different?

Split by era (pre-2020 vs post-2020) to separate the v2 normalization bug from
real hockey effects. If the period-2 anomaly exists only in the pre-2020 era, it
confirms the normalization bug. If it also exists post-2020, there may be a real
hockey or remaining data issue.

In [ ]:
PRE_2020_CUTOFF_SEASON = 20192020  # first season with homeTeamDefendingSide

cursor = conn.cursor()

cursor.execute("""
    SELECT
        CASE WHEN g.season < ? THEN 'pre-2020' ELSE 'post-2020' END AS era,
        se.period,
        COUNT(*) AS n_shots,
        AVG(se.x_coord) AS mean_x,
        AVG(se.distance_to_goal) AS mean_dist,
        SUM(CASE WHEN se.x_coord < 0 THEN 1 ELSE 0 END) AS neg_x_count,
        ROUND(100.0 * SUM(CASE WHEN se.x_coord < 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS neg_x_pct,
        SUM(se.is_goal) AS goals,
        ROUND(100.0 * SUM(se.is_goal) / COUNT(*), 2) AS goal_pct
    FROM shot_events se
    JOIN games g ON se.game_id = g.game_id
    WHERE (g.home_team_id = ? OR g.away_team_id = ?)
      AND se.shooting_team_id = ?
      AND se.period IN (1, 2, 3)
      AND se.x_coord IS NOT NULL
    GROUP BY era, se.period
    ORDER BY era, se.period
""", (PRE_2020_CUTOFF_SEASON, CAPS_TEAM_ID, CAPS_TEAM_ID, CAPS_TEAM_ID))

print(f"{'Era':<12} {'Period':>6} {'Shots':>7} {'Mean X':>8} {'Mean Dist':>10} "
      f"{'Neg X':>6} {'Neg X%':>7} {'Goals':>6} {'Goal%':>7}")
print("-" * 85)
for row in cursor.fetchall():
    print(f"{row[0]:<12} {row[1]:>6} {row[2]:>7} {row[3]:>8.1f} {row[4]:>10.1f} "
          f"{row[5]:>6} {row[6]:>6.1f}% {row[7]:>6} {row[8]:>6.2f}%")

In [ ]:
# ── KS test and Cohen's d: P2 vs P1+P3 distance distributions ─────────

def cohens_d(group1, group2):
    """Compute Cohen's d for two independent groups."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    if pooled_std == 0:
        return 0.0
    return (np.mean(group1) - np.mean(group2)) / pooled_std


def load_distances_by_period_era(era_condition, era_params):
    """Load distances for P1+P3 and P2 separately."""
    c = conn.cursor()
    p13 = []
    p2 = []
    for period_group, target in [("(1, 3)", p13), ("(2)", p2)]:
        c.execute(f"""
            SELECT se.distance_to_goal
            FROM shot_events se
            JOIN games g ON se.game_id = g.game_id
            WHERE {era_condition}
              AND (g.home_team_id = ? OR g.away_team_id = ?)
              AND se.shooting_team_id = ?
              AND se.period IN {period_group}
              AND se.distance_to_goal IS NOT NULL
        """, era_params + (CAPS_TEAM_ID, CAPS_TEAM_ID, CAPS_TEAM_ID))
        target.extend([row[0] for row in c.fetchall()])
    return np.array(p13), np.array(p2)


for era_name, condition, params in [
    ("Post-2020", "g.season >= ?", (PRE_2020_CUTOFF_SEASON,)),
    ("Pre-2020", "g.season < ?", (PRE_2020_CUTOFF_SEASON,)),
]:
    p13_dists, p2_dists = load_distances_by_period_era(condition, params)

    ks_stat, ks_p = stats.ks_2samp(p13_dists, p2_dists)
    d = cohens_d(p2_dists, p13_dists)

    print(f"\n{'=' * 60}")
    print(f"  {era_name}: Period 2 vs Periods 1+3 (Caps shots only)")
    print(f"{'=' * 60}")
    print(f"  P1+P3: n={len(p13_dists):,}, mean={np.mean(p13_dists):.2f}, median={np.median(p13_dists):.2f}")
    print(f"  P2:    n={len(p2_dists):,}, mean={np.mean(p2_dists):.2f}, median={np.median(p2_dists):.2f}")
    print(f"  Difference in means: {np.mean(p2_dists) - np.mean(p13_dists):.2f} ft")
    print(f"  KS test: statistic={ks_stat:.4f}, p={ks_p:.2e}")
    print(f"  Cohen's d: {d:.4f}")
    print(f"  Interpretation: |d|<0.2 = negligible, 0.2-0.5 = small, 0.5-0.8 = medium, >0.8 = large")

In [ ]:
# ── Box plots: distance by period, faceted by era ─────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, (era_name, condition, params) in zip(axes, [
    ("Post-2020 (normalization correct)", "g.season >= ?", (PRE_2020_CUTOFF_SEASON,)),
    ("Pre-2020 (v2 bug: raw coordinates)", "g.season < ?", (PRE_2020_CUTOFF_SEASON,)),
]):
    c = conn.cursor()
    periods_data = {}
    for period in [1, 2, 3]:
        c.execute(f"""
            SELECT se.distance_to_goal
            FROM shot_events se
            JOIN games g ON se.game_id = g.game_id
            WHERE {condition}
              AND (g.home_team_id = ? OR g.away_team_id = ?)
              AND se.shooting_team_id = ?
              AND se.period = ?
              AND se.distance_to_goal IS NOT NULL
        """, params + (CAPS_TEAM_ID, CAPS_TEAM_ID, CAPS_TEAM_ID, period))
        periods_data[period] = [row[0] for row in c.fetchall()]

    bp_data = [periods_data[1], periods_data[2], periods_data[3]]
    bp = ax.boxplot(bp_data, tick_labels=["P1", "P2", "P3"], patch_artist=True,
                    showfliers=False, widths=0.6)
    for patch, color in zip(bp["boxes"], ["#1f77b4", "#ff7f0e", "#2ca02c"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)

    # Add mean markers
    means = [np.mean(d) for d in bp_data]
    ax.scatter([1, 2, 3], means, color="red", zorder=5, s=50, marker="D", label="Mean")

    ax.set_title(era_name, fontsize=12)
    ax.set_ylabel("Distance to goal (ft)")
    ax.set_xlabel("Period")
    ax.legend(fontsize=9)

    # Annotate means
    for i, m in enumerate(means):
        ax.annotate(f"{m:.1f}", (i + 1, m), textcoords="offset points",
                    xytext=(15, 5), fontsize=9, color="red")

fig.suptitle("Caps Shot Distance by Period — Era Comparison", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
cursor = conn.cursor()
cursor.execute("""
    SELECT
        g.season,
        se.period,
        COUNT(*) AS total,
        SUM(CASE WHEN se.x_coord < 0 THEN 1 ELSE 0 END) AS neg_x,
        ROUND(100.0 * SUM(CASE WHEN se.x_coord < 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS neg_pct
    FROM shot_events se
    JOIN games g ON se.game_id = g.game_id
    WHERE (g.home_team_id = ? OR g.away_team_id = ?)
      AND se.shooting_team_id = ?
      AND se.period IN (1, 2, 3)
      AND se.x_coord IS NOT NULL
    GROUP BY g.season, se.period
    ORDER BY g.season, se.period
""", (CAPS_TEAM_ID, CAPS_TEAM_ID, CAPS_TEAM_ID))

rows = cursor.fetchall()

seasons = sorted(set(r[0] for r in rows))
neg_pct_by_season_period = {}
for r in rows:
    neg_pct_by_season_period[(r[0], r[1])] = r[4]

print(f"{'Season':<12} {'P1 neg%':>8} {'P2 neg%':>8} {'P3 neg%':>8}   Note")
print("-" * 60)
for season in seasons:
    p1 = neg_pct_by_season_period.get((season, 1), 0)
    p2 = neg_pct_by_season_period.get((season, 2), 0)
    p3 = neg_pct_by_season_period.get((season, 3), 0)
    note = ""
    if p1 > 10:
        note = "<-- v2 bug (no homeTeamDefendingSide)"
    elif p1 > 5:
        note = "<-- transition season"
    print(f"{season:<12} {p1:>7.1f}% {p2:>7.1f}% {p3:>7.1f}%   {note}")

In [ ]:
# ── Visual: neg x % by season, showing the era break ─────────────────

fig, ax = plt.subplots(figsize=(14, 5))

season_labels = [f"{str(s)[:4]}-{str(s)[4:]}" for s in seasons]
for period, color, label in [(1, "#1f77b4", "P1"), (2, "#ff7f0e", "P2"), (3, "#2ca02c", "P3")]:
    pcts = [neg_pct_by_season_period.get((s, period), 0) for s in seasons]
    ax.plot(season_labels, pcts, "o-", color=color, label=label, markersize=5)

ax.axvline(x=season_labels.index("2019-2020"), color="red", ls="--", lw=2, alpha=0.7,
           label="API change (homeTeamDefendingSide added)")
ax.set_xlabel("Season")
ax.set_ylabel("% shots with negative x_coord")
ax.set_title("Caps: Negative x_coord Rate by Season and Period")
ax.legend(fontsize=9)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

---
## Summary and Recommendations

### What we found

**1. Pre-2020 data (~1.3M shots, seasons 2007-08 through 2018-19) has a severe normalization bug.**

The NHL API does not provide `homeTeamDefendingSide` for these older games. The v2 extraction
code stored raw coordinates unchanged, so approximately half of all shots have their coordinates
on the wrong side of the rink. These shots show:
- Negative x_coord (should be positive after normalization)
- Average distance to goal ~150 ft (should be ~35 ft)
- Goal rate ~7% at those recorded distances (physically impossible — confirms the coords are wrong)

The negative x rate is **~50% across all three periods equally** for pre-2020 data. This means
the issue is **not** a period-2-specific direction-change problem — it's a blanket failure to
normalize when the API field is absent.

**2. Post-2020 data (~750k shots, seasons 2019-20 onward) appears correctly normalized.**

The API provides `homeTeamDefendingSide`, which alternates between periods (e.g., right → left → right).
The normalization handles this correctly:
- Only ~2% of shots have negative x (legitimate behind-center-ice events)
- Period 1, 2, and 3 distributions are very similar
- The KS test and Cohen's d will quantify whether any small difference exists

**3. A v3 backfill has never been run.**

The current code (`_XG_EVENT_SCHEMA_VERSION = "v3"`) includes a fallback heuristic for missing
`homeTeamDefendingSide`: if raw x < 0, flip to positive. This was committed in `2c1f745` but
the data is still all at v2. Running `backfill_missing_game_data()` will reprocess all games.

### Recommendations

1. **Run the v3 backfill** to reprocess all shot events. This will fix the majority of pre-2020
   shots by applying the sign-based heuristic. Note that the heuristic is imperfect: it correctly
   handles ~96% of shots (those near the attacking goal) but misclassifies shots from behind
   center ice (~4% of shots, low-danger events).

2. **For model training, consider era-aware handling**: either exclude pre-2020 data, or use
   the v3 heuristic with awareness that a small fraction of pre-2020 distances are still wrong.

3. **The period-2 concern specifically is not supported** for post-2020 data. The direction
   change is correctly handled by the `homeTeamDefendingSide` field from the API.

### Limitations of the v3 heuristic for pre-2020 data

The fallback `if x < 0: flip` works because most shots are taken in the offensive zone. But it
cannot distinguish:
- A shot near the attacking goal at x=70 (correct, keep) from
- A shot behind center ice at x=70 when the team attacks toward -x (wrong, should flip)

This affects ~4% of shots. These are predominantly long-range, low-danger events that have
minimal impact on xG modeling but should be documented as a known limitation.